In [ ]:
# import  os
# num_cores = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = num_cores
# os.environ["OMP_NUM_THREADS"] = num_cores
# os.environ["MKL_NUM_THREADS"] = num_cores

In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
from scipy.sparse.linalg import eigsh
import utils_2Q_gate_zp as ut
from joblib import Parallel, delayed
from IPython.display import display, Math
# ut.set_fig_font() ### Set various sizes in plotting
import networkx as nx
from multiprocessing import Pool
import pandas as pd

In [2]:
truc1, truc_tot, charge_pick  = 300, 1000, True
truc_tot_2 = 400
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()

truc_list = np.arange(truc_tot_2)
hspace_full = hspace_full[:truc_tot_2]
eval_tot = eval_tot[:truc_tot_2]
n_theta0_dress = ut.truncate_2(n_theta0_dress, truc_list)
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
logi_state = ['0-0', '0-2', '2-0', '2-2']

drive_term = n_theta1_dress
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]
core_states = logi_state + ['5-0']

### Truncation Estimate

In [3]:
max_n_ij = np.max(np.abs(drive_term.full()))
A = 0.02
population_rate = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
population_rate_log = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
rabi_df = []
G = nx.DiGraph()
for i, s_i in enumerate(hspace_full):
    for j, s_j in enumerate(hspace_full):
        if i < j:
            n_ij = np.abs(drive_term[i, j])/max_n_ij
            delta = abs(W_20_50 - (eval_tot[j] - eval_tot[i]))

            population_rate[i,j] = ((A*n_ij)**2) / ((A*n_ij)**2 + delta**2) 
            if population_rate[i,j] > 0:
                population_rate_log[i, j] = -np.log(population_rate[i,j])
                G.add_edge(s_i, s_j, weight=population_rate_log[i, j]) # Construct the graph

    rabi_df.append({"order": i, "i": s_i})
rabi_df = pd.DataFrame(rabi_df)
rabi_df.index = rabi_df["i"]
print('shape(population_rate_log)=', np.shape(population_rate_log))
# rabi_df

shape(population_rate_log)= (1000, 1000)


In [4]:
def shortest_path_to_core(target):
    shortest_path = ""
    shortest_path_len = np.inf
    for source in core_states[:-1]:
        if nx.has_path(G, source, target):
            path = nx.shortest_path(G, source=source, target=target,
                                    weight="weight")
            path_len = nx.shortest_path_length(G, source=source, target=target,
                                                weight="weight")
        if path_len < shortest_path_len:
            shortest_path_len = path_len
            shortest_path = ",".join([str(x) for x in path])
    return target, (shortest_path_len, shortest_path)

cutoff = 2
def all_path_to_core(target):
    path_tot = []
    if target in core_states:
        weight_tot = 1
    else:
        weight_tot = 0
        for source in core_states:
            for path in nx.all_simple_paths(G, source, target, cutoff=cutoff):
                weight_tot += np.exp( - nx.path_weight(G, path,'weight') )
                path_tot.append(path)
    return target, (weight_tot, path_tot)


### All path

In [5]:
# Find all_path_to_core
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(all_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = path[0]

df_all = rabi_df.sort_values("path_len", ascending=False)
print('all_path -- cutoff=', cutoff)
df_all.iloc[:30]

 40%|████      | 400/1000 [00:00<00:01, 519.54it/s]


all_path -- cutoff= 2


,order,i,path,path_len
i,,,,
0-0,0,0-0,[],1.000000e+00+0.000000e+ 00j
5-0,10,5-0,[],1.000000e+00+0.000000e+ 00j
0-2,3,0-2,[],1.000000e+00+0.000000e+ 00j
2-0,4,2-0,[],1.000000e+00+0.000000e+ 00j
2-2,13,2-2,[],1.000000e+00+0.000000e+ 00j
5-2,27,5-2,"[[0-0, 0-1, 5-2], [0-0, 1-0, 5-2], [0-0, 0-2, ...",2.388111e-02+0.000000e+ 00j
5-1,21,5-1,"[[0-0, 0-1, 5-1], [0-0, 1-0, 5-1], [0-0, 0-2, ...",6.786273e-03+0.000000e+ 00j
0-1,1,0-1,"[[0-0, 0-1]]",1.736684e-03+0.000000e+ 00j
2-1,9,2-1,"[[0-0, 0-1, 2-1], [0-0, 1-0, 2-1], [0-0, 0-2, ...",1.425617e-03+0.000000e+ 00j


In [6]:
print(f' all_{truc_tot_2} :')
data = df_all['i'].to_numpy().tolist()
dim = 10
for i in range(0, len(data), dim):  # Step size of 10
    if i%50==0:
        print('')
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

 all_400 :

'0-0', '5-0', '0-2', '2-0', '2-2', '5-2', '5-1', '0-1', '2-1', '1-0' ,
'0-5', '2-5', '1-2', '9-0', '4-0', '2-4', '5-5', '1-1', '5-4', '1-5' ,
'9-2', '0-4', '4-2', '2-8', '0-8', '9-1', '2-12', '2-9', '0-9', '0-12' ,
'12-0', '2-16', '0-16', '5-8', '1-8', '4-5', '2-13', '2-21', '0-18', '2-20' ,
'9-4', '4-9', '8-0', '2-18', '0-21', '2-24', '1-4', '18-0', '5-26', '0-13' ,

'13-0', '0-26', '15-0', '2-26', '1-12', '5-16', '8-1', '0-24', '15-1', '2-35' ,
'15-4', '2-33', '2-45', '0-33', '2-39', '0-45', '5-12', '0-39', '5-9', '8-12' ,
'5-33', '12-2', '4-4', '1-9', '4-1', '5-34', '2-30', '2-46', '1-16', '0-34' ,
'2-34', '8-2', '5-21', '2-52', '0-52', '0-42', '2-42', '2-59', '0-59', '5-18' ,
'0-65', '5-24', '2-55', '0-55', '0-20', '9-8', '8-9', '22-0', '1-25', '8-5' ,

'12-1', '4-8', '2-53', '2-36', '2-25', '5-20', '5-13', '9-24', '15-8', '1-30' ,
'0-25', '1-20', '5-25', '9-5', '1-13', '1-24', '13-2', '1-33', '18-1', '0-68' ,
'18-2', '20-0', '2-44', '9-12', '4-25', '9-9', '5-30', '0-83

In [7]:
# num_state = 200
# df_all_truc = df_all.iloc[:num_state].sort_values("order", ascending=True)
# print(f'state_all ({num_state}/{truc_tot_2}) :')
# data = df_all_truc['i']
# dim = 10
# for i in range(0, len(data), dim):  # Step size of 10
#     print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

# index_all = [hspace_full.index(i) for i in df_all_truc['i']]
# print(index_all)

### Shortest path

In [8]:
# Find shortest path for each node
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(shortest_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = np.exp(-path[0])

df_short = rabi_df.sort_values("path_len", ascending=False)
print('shortest_path:')
df_short.iloc[:30]

 40%|████      | 400/1000 [00:00<00:00, 876.08it/s] 

shortest_path:


,order,i,path,path_len
i,,,,
0-0,0,0-0,0-0,1.000000e+00+0.000000e+ 00j
5-0,10,5-0,"2-0,5-0",1.000000e+00-0.000000e+ 00j
0-2,3,0-2,0-2,1.000000e+00+0.000000e+ 00j
2-0,4,2-0,2-0,1.000000e+00+0.000000e+ 00j
2-2,13,2-2,2-2,1.000000e+00+0.000000e+ 00j
5-2,27,5-2,"2-2,5-2",2.388111e-02-0.000000e+ 00j
5-1,21,5-1,"2-0,5-0,5-1",3.393052e-03-0.000000e+ 00j
0-1,1,0-1,"0-0,0-1",1.736684e-03-0.000000e+ 00j
2-1,9,2-1,"2-0,2-1",1.425617e-03-0.000000e+ 00j


In [9]:
print(f' short_{truc_tot_2} :')
data = df_short['i'].to_numpy().tolist()
dim = 10
for i in range(0, len(data), dim):  # Step size of 10
    if i%50==0:
        print('')
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

 short_400 :

'0-0', '5-0', '0-2', '2-0', '2-2', '5-2', '5-1', '0-1', '2-1', '1-0' ,
'0-5', '2-5', '1-2', '9-0', '4-0', '5-5', '2-4', '1-1', '5-4', '1-5' ,
'9-2', '0-4', '4-2', '4-5', '0-8', '2-8', '2-12', '9-1', '0-9', '2-9' ,
'4-1', '0-12', '0-16', '2-16', '15-4', '1-4', '8-0', '12-0', '2-18', '9-5' ,
'2-13', '5-8', '1-8', '9-4', '2-21', '1-12', '5-9', '0-18', '0-21', '4-9' ,

'1-9', '2-20', '18-0', '5-12', '25-0', '0-13', '2-24', '0-26', '5-26', '2-26' ,
'13-0', '5-16', '15-0', '0-24', '1-18', '8-5', '2-45', '2-39', '8-1', '0-45' ,
'0-39', '8-12', '12-2', '15-1', '2-35', '0-33', '2-33', '8-2', '0-34', '5-33' ,
'1-16', '4-4', '2-34', '5-21', '5-34', '0-52', '2-52', '2-30', '2-46', '15-8' ,
'4-12', '0-42', '2-42', '2-59', '0-59', '0-65', '0-55', '2-55', '5-18', '1-25' ,

'8-9', '22-0', '18-1', '5-13', '4-8', '0-20', '9-8', '12-1', '5-24', '1-30' ,
'5-20', '5-25', '2-53', '1-13', '9-24', '0-25', '13-2', '2-25', '1-20', '18-2' ,
'0-68', '2-36', '15-2', '20-0', '1-24', '4-16', '18-5', '9

In [10]:
# num_state = 200
# df_short_truc = df_short.iloc[:num_state].sort_values("order", ascending=True)
# print('state_short :')
# data = df_short_truc['i']
# for i in range(0, len(data), dim):  # Step size of 10
#     print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

# index_all = [hspace_full.index(i) for i in df_short_truc['i']]
# print(index_all)

In [11]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [ ]:
all_82 = [
'0-0', '0-2', '2-0', '8-2', '2-2', '12-2', '4-9', '1-2', '1-0', '5-2' ,
'5-0', '8-5', '22-2', '9-2', '2-1', '5-8', '5-5', '0-1', '34-2', '2-5' ,
'13-2', '4-2', '8-0', '1-1', '0-5', '8-9', '15-2', '26-2', '30-2', '1-4' ,
'9-0', '20-2', '20-5', '12-5', '4-0', '15-5', '5-1', '18-2', '12-0', '24-2' ,
'26-5', '1-5', '15-0', '13-0', '25-2', '35-2', '8-1', '33-2', '9-5', '4-5' ,

'12-8', '9-8', '1-8', '37-2', '5-12', '25-0', '9-4', '1-25', '22-5', '56-2' ,
'18-4', '13-9', '4-4', '2-4', '15-4', '50-2', '13-5', '9-9', '5-9', '4-1' ,
'20-0', '25-5', '15-9', '18-5', '9-1', '30-5', '2-21', '26-9', '1-13', '20-9' ,
'45-2', '18-0', '5-16', '22-0', '34-5', '15-1', '5-4', '0-21', '20-4', '26-0' ,
'25-4', '25-1', '0-4', '9-12', '24-5', '37-5', '24-0', '1-18', '12-9', '44-2' ,

'25-8', '2-12', '41-2', '4-16', '4-12', '4-21', '8-12', '18-9', '18-1', '45-0' ,
'24-9', '26-8', '39-2', '4-8', '8-4', '34-0', '22-8', '35-4', '12-4', '30-0' ,
'12-1', '28-2', '18-8', '0-8', '35-5', '35-0', '9-16', '37-0', '2-8', '1-9' ,
'13-8', '46-2', '2-26', '46-0', '2-9', '41-0', '4-13', '39-0', '2-25', '0-12' ,
'22-9', '33-0', '33-4', '22-4', '54-2', '8-18', '28-1', '0-45', '50-0', '24-1' ,

'28-5', '30-1', '15-8', '34-1', '22-1', '20-1', '13-16', '12-18', '0-9', '24-8' ,
'13-1', '51-2', '33-5', '2-30', '1-21', '4-18', '59-0', '15-12', '0-25', '8-24' ,
'8-8', '9-18', '44-0', '26-1', '65-0', '56-0', '41-1', '41-4', '5-30', '39-4' ,
'30-4', '58-0', '5-13', '18-12', '0-30', '45-4', '0-39', '13-4', '46-1', '28-0' ,
'2-18', '0-24', '0-13', '9-13', '30-8', '69-0', '41-5', '33-1', '37-8', '4-24' ,
]
short_82 = [
'8-2', '0-0', '0-2', '2-0', '2-2', '12-2', '1-2', '1-0', '5-2', '5-0' ,
'4-9', '8-5', '2-1', '22-2', '9-2', '0-1', '5-8', '2-5', '4-2', '5-5' ,
'34-2', '8-0', '1-1', '0-5', '13-2', '8-9', '15-2', '26-2', '1-4', '9-0' ,
'20-2', '4-0', '30-2', '5-1', '12-5', '20-5', '12-0', '18-2', '15-5', '15-0' ,
'1-5', '13-0', '35-2', '33-2', '8-1', '24-2', '26-5', '4-5', '25-2', '4-1' ,

'12-8', '9-5', '9-8', '1-8', '5-12', '25-0', '12-9', '22-5', '56-2', '18-0' ,
'37-2', '1-25', '2-4', '4-4', '50-2', '18-4', '9-4', '5-9', '0-21', '15-4' ,
'20-0', '25-5', '13-5', '9-1', '46-2', '13-9', '30-5', '15-9', '9-9', '2-21' ,
'35-5', '5-4', '1-13', '26-9', '33-5', '20-9', '18-5', '22-0', '54-2', '34-5' ,
'26-0', '45-2', '20-4', '5-16', '0-4', '37-5', '15-1', '13-1', '24-5', '24-0' ,

'25-1', '4-8', '51-2', '12-1', '25-8', '25-4', '9-12', '13-12', '22-9', '4-21' ,
'45-0', '4-16', '41-2', '18-9', '1-18', '18-8', '44-2', '8-12', '8-4', '2-12' ,
'22-8', '26-8', '4-12', '0-8', '30-0', '41-0', '18-1', '4-25', '15-8', '24-9' ,
'37-0', '39-2', '35-4', '2-8', '1-9', '5-21', '35-0', '34-0', '33-0', '12-12' ,
'12-4', '46-0', '13-8', '2-9', '20-8', '28-2', '30-4', '9-16', '5-18', '39-0' ,

'0-12', '2-26', '12-20', '28-1', '0-45', '33-4', '8-18', '50-0', '8-21', '4-30' ,
'24-1', '0-9', '22-1', '2-25', '4-13', '22-4', '24-8', '30-1', '13-16', '34-1' ,
'1-21', '0-18', '41-5', '12-18', '8-16', '20-1', '59-0', '15-12', '25-9', '28-5' ,
'4-18', '39-5', '44-0', '65-0', '12-24', '56-0', '44-4', '8-24', '9-18', '30-9' ,
'41-4', '58-0', '39-4', '2-30', '28-4', '5-30', '8-8', '5-13', '12-13', '0-25' ,
]

list1 = all_82
list2 = short_82
common_elements = [item for item in list1 if item in list2]
only_in_list1 = [item for item in list1 if item not in list2]
only_in_list2 = [item for item in list2 if item not in list1]
unique_elements = only_in_list1 + only_in_list2

print(f" Only in 1:", len(only_in_list1), only_in_list1)
print('index only in 1:', [list1.index(i) for i in only_in_list1])
print("Only in 2:", len(only_in_list2), only_in_list2)
print('index only in 2:', [list2.index(i) for i in only_in_list2])

 Only in 1: 18 ['26-1', '41-1', '18-12', '0-30', '45-4', '0-39', '13-4', '46-1', '28-0', '2-18', '0-24', '0-13', '9-13', '30-8', '69-0', '33-1', '37-8', '4-24']
index only in 1: [173, 176, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 197, 198, 199]
Only in 2: 18 ['13-12', '4-25', '5-21', '12-12', '20-8', '5-18', '12-20', '8-21', '4-30', '0-18', '8-16', '25-9', '39-5', '12-24', '44-4', '30-9', '28-4', '12-13']
index only in 2: [107, 127, 135, 139, 144, 148, 152, 158, 159, 171, 174, 178, 181, 184, 186, 189, 194, 198]
